In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from apex import amp
from torch.optim import AdamW
from torch.nn import MSELoss, BCEWithLogitsLoss
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from source.version1.data import trainLoader
from source.version1.model import EfficientModel
from source.version1.train import trainModel
from source.version1.loss import BCELoss
from catalyst.data.sampler import BalanceClassSampler

In [3]:
SEED = 42

def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(42)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/combined/train/train/'
    loader['label_path'] = '../../data/combined/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 10
    params['num_workers'] = 3
    params['drop_last'] = True
    sampler = BalanceClassSampler(labels = train.labels(), mode="downsampling")
    train = DataLoader(train, sampler = sampler, **params)
    valid = DataLoader(valid, **params)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = AdamW(model.parameters(), lr=3e-05, weight_decay=0.)
    schedular = ReduceLROnPlateau(optimizer, factor=0.8, patience=0, min_lr=1e-8)
    model, optimizer = amp.initialize(model, optimizer, opt_level='O2', verbosity=False)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = BCELoss() 
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version1/model_{}.pt'.format(fold)
    trainer['epochs'] = 20
    trainer['batch'] = 10
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(0)

Train Images: 45780 Valid Images: 6478
Loaded pretrained weights for efficientnet-b5


100% 7740/7740 [08:21<00:00, 15.44it/s, trn_ls=0.5282, val_ls=0.2961, val_mt=0.9050]
 92% 7150/7740 [06:29<00:31, 18.64it/s, trn_ls=0.47440]

In [ ]:
train(1)

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)